In [1]:
import os 

from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages form the PDF")
print("first page preview --- first 500 character")
print(pages[0].page_content[:500])

Loaded 9 pages form the PDF
first page preview --- first 500 character
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [4]:
print(pages[1].page_content[:500])

Telecom Technical Reference Guide  - Internal Use Only
1. Introduction to Mobile Networks
Mobile networks have evolved through several generations, each offering significant improvements in speed,
capacity, and capability.
2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to
around 50 kbps, sufficient only for text messaging and simple email.
3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, an


In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap = 100,
    separators = ["\n\n", "\n", ".", " " ] 
)

chunks = splitter.split_documents(pages)

len(chunks)

37

In [31]:
chunks[0].page_content

'Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [32]:
chunks[1].page_content

'Telecom Technical Reference Guide  - Internal Use Only\n1. Introduction to Mobile Networks\nMobile networks have evolved through several generations, each offering significant improvements in speed,\ncapacity, and capability.\n2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to\naround 50 kbps, sufficient only for text messaging and simple email.\n3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, and app'

In [33]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2480.51it/s]


Vector store ready. 193 vectors stored


In [34]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

test_query = "What is VoLTE and how does it improve the call quality?"
retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved, 1):
    print(f"--- chunk {i} ---")
    print(doc.page_content[:300])
    print()

--- chunk 1 ---
legacy calls), faster call setup times (under 2 seconds versus 6-8 seconds on 3G), and the ability to use data and
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navig

--- chunk 2 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- chunk 3 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

